# Steam Game Analytics

Predict review sentiment from game features, then recommend similar games from genre, price, and popularity.

The cleaning rules, model parameters, train/test split, and recommendation weights match the original notebook. Repeated loads and overwritten helper functions are consolidated.


In [ ]:
import ast
import os
from collections import Counter
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler
from sklearn.tree import DecisionTreeClassifier

DATA_FILENAMES = [
    "games_march2025_cleaned.csv",
    "games_march2025_full.csv",
    "games_may2024_cleaned.csv",
    "games_may2024_full.csv",
]
PRIMARY_DATASET = "games_march2025_cleaned.csv"
SEARCH_DIRECTORIES = [Path.cwd(), Path("/content")]

IMPORTANT_COLUMNS = [
    "name",
    "release_date",
    "price",
    "genres",
    "positive",
    "negative",
    "metacritic_score",
    "average_playtime_forever",
    "pct_pos_total",
    "num_reviews_total",
]
REVIEW_COLUMNS = [
    "price",
    "positive",
    "negative",
    "metacritic_score",
    "average_playtime_forever",
    "pct_pos_total",
    "num_reviews_total",
]
NUMERIC_COLUMNS = [
    "price",
    "positive",
    "negative",
    "metacritic_score",
    "average_playtime_forever",
    "pct_pos_total",
]
FEATURE_COLUMNS = [
    "price",
    "required_age",
    "dlc_count",
    "windows",
    "mac",
    "linux",
    "achievements",
    "average_playtime_forever",
    "average_playtime_2weeks",
    "median_playtime_forever",
    "median_playtime_2weeks",
    "peak_ccu",
    "discount",
    "metacritic_score",
]
PLATFORM_COLUMNS = ["windows", "mac", "linux"]
SENTIMENT_LABELS = ["Negative", "Mixed", "Positive"]
RECOMMENDATION_COLUMNS = ["appid", "name", "genres", "price", "positive", "negative"]
MODEL_PATH = "steam_sentiment_model.pkl"

# Final recommendation weights from the last version of recommend_games.
GENRE_WEIGHT = 0.70
PRICE_WEIGHT = 0.15
POPULARITY_WEIGHT = 0.15


def find_dataset(filename):
    """Return the first existing copy of filename, or the local path if none exist."""
    for directory in SEARCH_DIRECTORIES:
        candidate = directory / filename
        if candidate.exists():
            return candidate
    return SEARCH_DIRECTORIES[0] / filename


def read_games_csv(path, on_bad_lines):
    return pd.read_csv(path, engine="python", on_bad_lines=on_bad_lines)


def classify_sentiment(score):
    """Map a 0-100 positive-review percentage to a sentiment label."""
    if score >= 70:
        return "Positive"
    if score >= 40:
        return "Mixed"
    return "Negative"


def parse_genres(value):
    """Turn a genres cell into a list. Invalid or empty cells become []."""
    if pd.isna(value):
        return []
    try:
        parsed = ast.literal_eval(value) if isinstance(value, str) else value
        return parsed if isinstance(parsed, list) else []
    except (ValueError, SyntaxError, TypeError):
        return []


def print_classification_results(
    y_true,
    y_pred,
    heading=None,
    accuracy_label="Accuracy:",
    show_confusion_matrix=False,
):
    """Print accuracy and the classification report in the original layout."""
    if heading:
        print(heading)
        print("-" * 35)

    print(accuracy_label, round(accuracy_score(y_true, y_pred), 4))
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, zero_division=0))

    if show_confusion_matrix:
        print("\nConfusion Matrix:")
        print(confusion_matrix(y_true, y_pred, labels=SENTIMENT_LABELS))


## Load the datasets

In [ ]:
for filename in DATA_FILENAMES:
    path = find_dataset(filename)
    if not path.exists():
        print("FILE NOT FOUND:", path)
        continue

    try:
        preview = read_games_csv(path, on_bad_lines="warn")
        print("\n" + "=" * 50)
        print("FILE:", path.name)
        print("Shape:", preview.shape)
        print("Columns:", preview.columns.tolist())
        print("Total missing values:", preview.isnull().sum().sum())
        print("Duplicate rows:", preview.duplicated().sum())
        display(preview.head(3))
    except Exception as error:
        print("Error in", path.name, ":", error)

dataset_path = find_dataset(PRIMARY_DATASET)
df = read_games_csv(dataset_path, on_bad_lines="skip")

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)
print("\nFirst 5 rows:")
display(df.head())
print("\nData types:")
print(df.dtypes)


## Inspect missing values and review fields

In [ ]:
print("Missing values in each column:")
print(df.isnull().sum().sort_values(ascending=False).head(20))
print("\nTotal missing values:", df.isnull().sum().sum())

print("\nMissing values in important columns:")
print(df[REVIEW_COLUMNS].isnull().sum())

print("\nMetacritic score values:")
print(df["metacritic_score"].value_counts().head(10))
print("\nPercentage score values:")
print(df["pct_pos_total"].value_counts().head(10))
print("\nReview count values:")
print(df["num_reviews_total"].value_counts().head(10))
print("\nDuplicate app IDs:", df["appid"].duplicated().sum())

print("\nSummary of important columns:")
print(df[IMPORTANT_COLUMNS].describe(include="all").T)

display(df[NUMERIC_COLUMNS].describe().T)


## Explore price, genre, and review patterns

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(data=df[df["price"].between(0, 100)], x="price", bins=30)
plt.title("Distribution of Steam Game Prices")
plt.xlabel("Price")
plt.ylabel("Number of Games")
plt.show()

genre_counts = Counter()
for value in df["genres"].dropna():
    genre_counts.update(parse_genres(value))

top_genres = genre_counts.most_common(10)
genre_df = pd.DataFrame(top_genres, columns=["Genre", "Count"])

plt.figure(figsize=(10, 6))
sns.barplot(data=genre_df, x="Count", y="Genre")
plt.title("Top 10 Steam Game Genres")
plt.xlabel("Number of Games")
plt.ylabel("Genre")
plt.show()

review_plot_df = df[(df["positive"] >= 0) & (df["negative"] >= 0)].copy()
plt.figure(figsize=(8, 5))
sns.scatterplot(
    data=review_plot_df.sample(min(3000, len(review_plot_df)), random_state=42),
    x="positive",
    y="negative",
    alpha=0.5,
)
plt.title("Positive Reviews vs Negative Reviews")
plt.xlabel("Positive Reviews")
plt.ylabel("Negative Reviews")
plt.show()


## Check the sentiment target

In [ ]:
print("Values below zero:")
print((df["pct_pos_total"] < 0).sum())
print("\nMissing values:")
print(df["pct_pos_total"].isnull().sum())
print("\nValid percentage summary:")
print(df.loc[df["pct_pos_total"].between(0, 100), "pct_pos_total"].describe())

print("\nDataset shape:", df.shape)
print("\nAll distinct pct_pos_total values (first 20):")
print(sorted(df["pct_pos_total"].unique())[:20])
print("\nValue counts:")
print(df["pct_pos_total"].value_counts().head(10))
print("\nCounts by condition:")
print("Below 0:", (df["pct_pos_total"] < 0).sum())
print("Equal to -1:", (df["pct_pos_total"] == -1).sum())
print("Between 0 and 100:", df["pct_pos_total"].between(0, 100).sum())
print("Above 100:", (df["pct_pos_total"] > 100).sum())
print("\nValid percentage rows:", df["pct_pos_total"].between(0, 100).sum())
print("Unavailable/special values:", (~df["pct_pos_total"].between(0, 100)).sum())


## Build the sentiment dataset and split

Games outside the 0–100 review-percentage range are excluded. Sentiment labels stay:

- Positive: 70 and above
- Mixed: 40 up to, but not including, 70
- Negative: below 40


In [ ]:
ml_df = df[df["pct_pos_total"].between(0, 100)].copy()
ml_df["sentiment"] = ml_df["pct_pos_total"].apply(classify_sentiment)

print("ML dataset shape:", ml_df.shape)
print("\nSentiment distribution:")
print(ml_df["sentiment"].value_counts())
print("\nSentiment percentages:")
print((ml_df["sentiment"].value_counts(normalize=True) * 100).round(2))

available_features = [column for column in FEATURE_COLUMNS if column in ml_df.columns]
print("\nSelected candidate features:", available_features)
print("\nMissing values in candidate features:")
print(ml_df[available_features].isnull().sum().sort_values(ascending=False))
print("\nTarget distribution:")
print(ml_df["sentiment"].value_counts())

X = ml_df[FEATURE_COLUMNS].copy()
y = ml_df["sentiment"].copy()
X[PLATFORM_COLUMNS] = X[PLATFORM_COLUMNS].astype(int)

print("\nX shape:", X.shape)
print("y shape:", y.shape)
print("\nFeature columns:")
print(X.columns.tolist())

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print("\nTraining rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])
print("\nTraining class distribution:")
print(y_train.value_counts(normalize=True).round(3))
print("\nTesting class distribution:")
print(y_test.value_counts(normalize=True).round(3))


## Train the same three classifiers

In [ ]:
logistic_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42,
    )),
])
logistic_model.fit(X_train, y_train)
y_pred_lr = logistic_model.predict(X_test)
print_classification_results(
    y_test,
    y_pred_lr,
    heading="Logistic Regression Results",
    show_confusion_matrix=True,
)

dt_model = DecisionTreeClassifier(
    max_depth=10,
    class_weight="balanced",
    random_state=42,
)
dt_model.fit(X_train, y_train)
y_pred_dt = dt_model.predict(X_test)
print_classification_results(
    y_test,
    y_pred_dt,
    heading="Decision Tree Results",
)

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)
print_classification_results(
    y_test,
    y_pred_rf,
    heading="Random Forest Results",
)


## Random Forest feature importance

In [ ]:
importance_df = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": rf_model.feature_importances_,
}).sort_values("Importance", ascending=False)

print(importance_df)

plt.figure(figsize=(9, 6))
sns.barplot(data=importance_df, x="Importance", y="Feature")
plt.title("Random Forest Feature Importance")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()


## Tune the Random Forest and save it

In [ ]:
rf_base = RandomForestClassifier(
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [10, 15, None],
    "min_samples_leaf": [2, 5],
}
grid_search = GridSearchCV(
    estimator=rf_base,
    param_grid=param_grid,
    scoring="f1_macro",
    cv=3,
    n_jobs=-1,
    verbose=1,
)
grid_search.fit(X_train, y_train)

print("Best Parameters:")
print(grid_search.best_params_)
print("\nBest Cross-Validation Macro F1:")
print(round(grid_search.best_score_, 4))

best_rf = grid_search.best_estimator_
y_pred_best = best_rf.predict(X_test)
print_classification_results(
    y_test,
    y_pred_best,
    accuracy_label="Tuned Random Forest Accuracy:",
    show_confusion_matrix=True,
)

joblib.dump(best_rf, MODEL_PATH)
print("\nModel saved successfully!")

loaded_model = joblib.load(MODEL_PATH)
print("Model loaded successfully!")


## Recommend similar games

Similarity is the final weighted score: 70% genre cosine similarity, 15% price closeness, and 15% log-popularity closeness. The selected game is excluded by setting its score to -1.


In [ ]:
def prepare_recommendation_data(games):
    """Keep one row per app and encode genres for similarity search."""
    prepared = games[RECOMMENDATION_COLUMNS].copy()
    prepared["genres_list"] = prepared["genres"].apply(parse_genres)
    prepared = prepared.dropna(subset=["name"])
    prepared = prepared.drop_duplicates(subset=["appid"])
    prepared = prepared.reset_index(drop=True)

    for column in ["price", "positive", "negative"]:
        prepared[column] = pd.to_numeric(prepared[column], errors="coerce").fillna(0)

    encoder = MultiLabelBinarizer()
    encoded_genres = encoder.fit_transform(prepared["genres_list"])
    return prepared, encoded_genres, encoder


def recommend_games(game_name, top_n=5):
    """Return the top_n games most similar to game_name."""
    matches = rec_df[rec_df["name"].str.casefold() == game_name.casefold()]
    if matches.empty:
        print("Game not found. Check the spelling.")
        return pd.DataFrame()

    selected_index = matches.index[0]
    genre_scores = cosine_similarity(
        genre_matrix[selected_index:selected_index + 1],
        genre_matrix,
    ).flatten()

    prices = rec_df["price"].clip(lower=0)
    selected_price = prices.iloc[selected_index]
    price_scores = (
        1 - (prices - selected_price).abs() / max(float(prices.max()), 1)
    ).clip(0, 1).to_numpy()

    popularity = np.log1p(rec_df["positive"].clip(lower=0))
    max_popularity = max(float(popularity.max()), 1)
    popularity_scores = (
        1 - (popularity - popularity.iloc[selected_index]).abs() / max_popularity
    ).clip(0, 1).to_numpy()

    scores = (
        GENRE_WEIGHT * genre_scores
        + PRICE_WEIGHT * price_scores
        + POPULARITY_WEIGHT * popularity_scores
    )
    scores[selected_index] = -1

    top_indices = np.argsort(scores)[-top_n:][::-1]
    result = rec_df.iloc[top_indices][
        ["name", "price", "genres", "positive", "negative"]
    ].copy()
    result["similarity"] = scores[top_indices].round(3)
    return result.reset_index(drop=True)


rec_df, genre_matrix, genre_encoder = prepare_recommendation_data(df)

print("Games available for recommendation:", len(rec_df))
print("Number of unique genres:", len(genre_encoder.classes_))
print("Genre matrix shape:", genre_matrix.shape)
display(rec_df[["name", "genres_list", "price"]].head())

recommendations = recommend_games("Counter-Strike 2", top_n=5)
display(recommendations)


## Explain recommendations with Groq

In [ ]:
# Optional: install the client used by the explanation cells.
# %pip install -q groq python-dotenv


In [ ]:
from getpass import getpass

if not os.getenv("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")


In [ ]:
from groq import Groq

client = Groq(api_key=os.environ["GROQ_API_KEY"])

response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {
            "role": "user",
            "content": "Explain a Steam game recommendation system in two simple sentences.",
        }
    ],
    max_completion_tokens=200,
)
print(response.choices[0].message.content)
print("Finish reason:", response.choices[0].finish_reason)
print("AI response:", repr(response.choices[0].message.content))

response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {
            "role": "user",
            "content": "Say hello in one short sentence.",
        }
    ],
    reasoning_effort="low",
    max_completion_tokens=1024,
)
print("Finish reason:", response.choices[0].finish_reason)
print("Response:", response.choices[0].message.content)


In [ ]:
def explain_recommendations(game_name, recommendations):
    """Ask Groq to explain an existing recommendation table. Do not invent features."""
    if recommendations is None or recommendations.empty:
        return "No recommendations available."

    games_text = recommendations[
        ["name", "price", "genres", "positive", "negative", "similarity"]
    ].to_string(index=False)

    prompt = f"""
You are a Steam game recommendation assistant.

Selected game: {game_name}

Recommended games:
{games_text}

Explain in simple English why these games were recommended.
Mention their shared genres, price, and similarity scores.
The similarity score is a custom ranking score, not a probability.
Use only the supplied information. Do not invent game features.
Keep the explanation concise and beginner-friendly.
"""

    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[{"role": "user", "content": prompt}],
        reasoning_effort="low",
        max_completion_tokens=1024,
    )
    return response.choices[0].message.content or (
        "The AI did not return an explanation. Please try again."
    )


game_name = "Counter-Strike 2"
recommendations = recommend_games(game_name, top_n=5)
display(recommendations)
print("\nAI Explanation:\n")
print(explain_recommendations(game_name, recommendations))


## Dashboard helpers

In [ ]:
%%writefile notebook_streamlit_stub.py
import streamlit as st

st.set_page_config(
    page_title="Steam Game Analytics",
    page_icon="🎮",
    layout="wide",
)

st.title("🎮 AI-Powered Steam Game Analytics")
st.write(
    "Discover similar Steam games using machine learning "
    "and AI-generated explanations."
)
st.success("Streamlit dashboard is running!")

st.subheader("Project Features")
st.markdown("""
- Game recommendations
- Genre and price comparison
- AI-generated explanations using Groq
- Steam game analytics and visualizations
""")

game_name = st.text_input(
    "Enter a Steam game name",
    value="Counter-Strike 2",
)
st.write("Selected game:", game_name)


In [ ]:
from dotenv import load_dotenv

load_dotenv()
print("Groq key available:", bool(os.getenv("GROQ_API_KEY")))
print("Current folder:", Path.cwd())
print("Dataset exists:", find_dataset(PRIMARY_DATASET).exists())


In [ ]:
from recommendation import load_recommendation_data

cleaned_data, module_genre_matrix = load_recommendation_data()

print("Cleaned dataset shape:", cleaned_data.shape)
print("Duplicate appids:", cleaned_data["appid"].duplicated().sum())
print("Missing game names:", cleaned_data["name"].isna().sum())
print("\nOriginal genre values:")
display(cleaned_data[["name", "genres"]].head())
print("\nParsed genre values:")
display(cleaned_data[["name", "genres_list"]].head())
print("\nGenre matrix shape:", module_genre_matrix.shape)
